<a href="https://colab.research.google.com/github/Luci893/prediccion-ventas-black-friday/blob/main/Trabajo_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aprendizaje Automático: Regresión Lineal

## Predicción de Ventas del Black Friday:

Vamos a utilizar un conjunto de datos de compras de productos durante un Black Friday (en EE. UU.). La idea principal es poder generar un predictor que nos permita predecir el `monto de la compra`.

Para lograr un buen predictor debemos aplicar los diferentes conceptos que hemos venido aprendiendo:

* `Exploración`
* `Ingeniería de Características` (Feature Engineering)
* `Modelado`
* `Evaluación`

El conjunto de datos aquí es una muestra de las transacciones realizadas en una tienda minorista. La tienda desea conocer mejor el comportamiento de `compra` de los clientes para diferentes productos. El problema es un `problema de regresión` donde intentamos predecir la variable dependiente (el monto de la compra) con la ayuda de la información contenida en las otras variables.

### Puedes probar diferentes modelos de Scikit-Learn desde [Modelos Lineales](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.linear_model)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

data = pd.read_csv("https://raw.githubusercontent.com/anyoneai/notebooks/main/datasets/BlackFriday.csv")
data.sample(5)

,User_ID,Product_ID,Gender,Age,Occupation,City_Category,Stay_In_Current_City_Years,Marital_Status,Product_Category_1,Product_Category_2,Product_Category_3,Purchase
473303,1000957,P00320442,M,36-45,1,C,2,1,6,10.0,NaN,20113
474798,1001159,P00174242,M,18-25,0,C,3,1,15,16.0,NaN,8760
360440,1001496,P00016042,M,26-35,17,A,2,0,1,16.0,NaN,15176
522905,1002537,P00184942,M,36-45,11,B,0,0,1,8.0,17.0,15786
65568,1004041,P00125342,M,18-25,3,B,2,0,11,NaN,NaN,4576


In [ ]:
# EXPLORACIÓN
data.info()

data.columns

print("\n")

for col in data.columns:
  print(f"Columna: {col}")
  print(f"Valores totales: {data[col].count()}")
  print(f"Valores únicos: {data[col].nunique()}")
  print(f"Valores nulos: {data[col].isnull().sum()}")
  print(f"Valores duplicados: {data[col].duplicated().sum()}")
  print(f"Tipo de dato: {data[col].dtype}")
  print("-" *30)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 537577 entries, 0 to 537576
Data columns (total 12 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   User_ID                     537577 non-null  int64  
 1   Product_ID                  537577 non-null  object 
 2   Gender                      537577 non-null  object 
 3   Age                         537577 non-null  object 
 4   Occupation                  537577 non-null  int64  
 5   City_Category               537577 non-null  object 
 6   Stay_In_Current_City_Years  537577 non-null  object 
 7   Marital_Status              537577 non-null  int64  
 8   Product_Category_1          537577 non-null  int64  
 9   Product_Category_2          370591 non-null  float64
 10  Product_Category_3          164278 non-null  float64
 11  Purchase                    537577 non-null  int64  
dtypes: float64(2), int64(5), object(5)
memory usage: 49.2+ MB


Columna: Use

In [ ]:
# FEATURE ENGINEERING

# Dejar columnas necesarias únicamente
data = data[['Gender', 'Age', 'Occupation', 'City_Category',
      'Stay_In_Current_City_Years', 'Marital_Status', 'Product_Category_1',
      'Product_Category_2', 'Product_Category_3', 'Purchase']]
data.head(7)

,Gender,Age,Occupation,City_Category,Stay_In_Current_City_Years,Marital_Status,Product_Category_1,Product_Category_2,Product_Category_3,Purchase
0,F,0-17,10,A,2,0,3,NaN,NaN,8370
1,F,0-17,10,A,2,0,1,6.0,14.0,15200
2,F,0-17,10,A,2,0,12,NaN,NaN,1422
3,F,0-17,10,A,2,0,12,14.0,NaN,1057
4,M,55+,16,C,4+,0,8,NaN,NaN,7969
5,M,26-35,15,A,3,0,1,2.0,NaN,15227
6,M,46-50,7,B,2,1,1,8.0,17.0,19215


In [ ]:
# Modificar nulos a valor 0
data['Product_Category_2'] = data['Product_Category_2'].fillna(0)
data['Product_Category_3'] = data['Product_Category_3'].fillna(0)

print(data.shape) # Verifico cuanto datos me quedan


(537577, 10)


In [ ]:
# Convertir variables categóricas a numéricas, con dytype=int lo fuerzo a que pase los valores a 0 y 1 en vez de true y false
data_dummies = pd.get_dummies(data, columns=['City_Category', 'Gender'], dtype=int)

# Transformar rango de AGE a un valor promedio
age_mapping = {
    '0-17': 8,
    '18-25': 21.5,
    '26-35': 30.5,
    '36-45': 40.5,
    '46-50': 48,
    '51-55': 53.5,
    '55+': 60}
# mapeo el array en la columna age
data_dummies['Age'] = data_dummies['Age'].map(age_mapping)

# Transformar 4+ de la columna 'Stay_In_Current_City_Years' a 5
data_dummies['Stay_In_Current_City_Years'] = data_dummies['Stay_In_Current_City_Years'].replace('4+', '5').astype(int)

# Crear nueva columna de cantidad de categoria en que esta el producto
data_dummies['num_categorias'] = (data_dummies[['Product_Category_1', 'Product_Category_2', 'Product_Category_3']] != 0).sum(axis=1)

data_dummies.head(7)

,Age,Occupation,Stay_In_Current_City_Years,Marital_Status,Product_Category_1,Product_Category_2,Product_Category_3,Purchase,City_Category_A,City_Category_B,City_Category_C,Gender_F,Gender_M,num_categorias
0,8.0,10,2,0,3,0.0,0.0,8370,1,0,0,1,0,1
1,8.0,10,2,0,1,6.0,14.0,15200,1,0,0,1,0,3
2,8.0,10,2,0,12,0.0,0.0,1422,1,0,0,1,0,1
3,8.0,10,2,0,12,14.0,0.0,1057,1,0,0,1,0,2
4,60.0,16,5,0,8,0.0,0.0,7969,0,0,1,0,1,1
5,30.5,15,3,0,1,2.0,0.0,15227,1,0,0,0,1,2
6,48.0,7,2,1,1,8.0,17.0,19215,0,1,0,0,1,3


In [ ]:
# MODELADO

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Preparar daos para regresión(predecir importe de compras)

X_reg = data_dummies.drop('Purchase', axis=1)
Y_reg = data_dummies['Purchase']

# X_reg -> Variables indepiendientes
# Y_reg -> Variable objetivo
# X_reg_train -> Datos de entrada para entrenar (80% de los datos)
# Y_reg_train -> Respuestas Correctas para entrenar (80% de los datos)
# X_reg_test -> Datos de entrada para probar (20% de los datos)
# Y_reg_test -> Respuestas reales para comparar (20%)

# test_size=0.2 -> significa 80% de entrenamiento y 20% de test
# random_state -> hace que la división sea siempre la misma

# Dividir en conjunto de entrenamiento y prueba
X_reg_train, X_reg_test, Y_reg_train, Y_reg_test = train_test_split(X_reg, Y_reg, test_size=0.2, random_state=42)

print(X_reg_train.shape)
print(X_reg_test.shape)
print(Y_reg_train.shape)
print(Y_reg_test.shape)


(430061, 13)
(107516, 13)
(430061,)
(107516,)


In [ ]:
# Modelo de regresión
reg_model = LinearRegression()
reg_model.fit(X_reg_train, Y_reg_train) # Variables para predecir


LinearRegression()

In [ ]:
# Predecir
Y_reg_pred = reg_model.predict(X_reg_test)
Y_reg_pred

array([12936.00120583,  9720.67039951,  7552.17125928, ...,
        8904.22295298,  8849.61545595,  9433.3184459 ])

In [ ]:
#Indices randoms
random_index = np.random.randint(0, len(X_reg_test))

random_ejemplo = X_reg_test.iloc[[random_index]] # Selecciona una fila especifica
actual_value = Y_reg_test.iloc[random_index] # Obtiene el valor real de esa fila

predicted_value = reg_model.predict(random_ejemplo)

print(f"Predicción para una entrada aleatoria:\n")
print(f"Características de la entrada:\n{random_ejemplo.to_string()}\n")
print(f"Valor Real (Purchase): {actual_value:.2f}")
print(f"Valor Predicho (Purchase): {predicted_value[0]:.2f}")

Predicción para una entrada aleatoria:

Características de la entrada:
         Age  Occupation  Stay_In_Current_City_Years  Marital_Status  Product_Category_1  Product_Category_2  Product_Category_3  City_Category_A  City_Category_B  City_Category_C  Gender_F  Gender_M  num_categorias
461354  30.5          17                           1               1                   2                 4.0                 9.0                0                1                0         0         1               3

Valor Real (Purchase): 6695.00
Valor Predicho (Purchase): 11701.90


In [ ]:
# EVALUACIÓN
from sklearn.metrics import mean_squared_error

print(data_dummies['Purchase'].mean())

rmse = np.sqrt(mean_squared_error(Y_reg_test, Y_reg_pred))
print(f'Regresión - RMSE: {rmse:.2f}')

# Este resultado indicaría que hay un error en la predición del importe de aproximadamente 4620 unidades y comparandolo con la media este resultado es moderado.

9333.859852635065
Regresión - RMSE: 4620.05
